# Practice 3: Customer Clustering - Bước 2: Kiểm tra dữ liệu (Data Collection & Validation)

---

## 1. Import các thư viện cần thiết

Chúng ta bắt đầu bằng việc import các thư viện cốt lõi để thao tác dữ liệu: `pandas` và `numpy`.

In [1]:
import pandas as pd
import numpy as np
import os

## 2. Tải tập dữ liệu thô (Raw Data)

Đường dẫn đến thư mục dữ liệu gốc là `../data/raw/`.

In [2]:
train_path = "../data/raw/Train.csv"
test_path = "../data/raw/Test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print(f"Kích thước tập Train: {train_df.shape}")
print(f"Kích thước tập Test:  {test_df.shape}")

Kích thước tập Train: (8068, 11)
Kích thước tập Test:  (2627, 10)


## 3. Xem trước một số dòng dữ liệu

Quan sát 20 dòng đầu tiên của tập Train để hình dung cấu trúc các đặc trưng.

In [9]:
train_df.head(20)

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A
5,461319,Male,Yes,56,No,Artist,0.0,Average,2.0,Cat_6,C
6,460156,Male,No,32,Yes,Healthcare,1.0,Low,3.0,Cat_6,C
7,464347,Female,No,33,Yes,Healthcare,1.0,Low,3.0,Cat_6,D
8,465015,Female,Yes,61,Yes,Engineer,0.0,Low,3.0,Cat_7,D
9,465176,Female,Yes,55,Yes,Artist,1.0,Average,4.0,Cat_6,C


## 4. Kiểm tra kiểu dữ liệu của các đặc trưng (Data Types)

Xác định các cột nào là kiểu số (numerical) và cột nào là kiểu phân loại (categorical).

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


## 5. Kiểm tra giá trị khuyết thiếu (Missing Values)

Thống kê số lượng và tỷ lệ % khuyết thiếu trên mỗi cột để lập phương án điền khuyết ở bước sau.

In [5]:
def check_missing_values(df):
    missing_count = df.isnull().sum()
    missing_percent = (df.isnull().sum() / len(df)) * 100
    missing_table = pd.concat([missing_count, missing_percent], axis=1, keys=['Missing Count', 'Percentage (%)'])
    return missing_table[missing_table['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

print("--- Các cột bị khuyết thiếu ở tập Train ---")
print(check_missing_values(train_df))

--- Các cột bị khuyết thiếu ở tập Train ---
                 Missing Count  Percentage (%)
Work_Experience            829       10.275161
Family_Size                335        4.152206
Ever_Married               140        1.735250
Profession                 124        1.536936
Graduated                   78        0.966782
Var_1                       76        0.941993


## 6. Kiểm tra dòng trùng lặp (Duplicate Rows)

Trùng lặp dữ liệu có thể làm sai lệch phân phối và kết quả phân cụm. Ta kiểm tra xem có dòng nào trùng hoàn toàn (hoặc trùng mã định danh `ID`) hay không.

In [6]:
duplicates_total = train_df.duplicated().sum()
duplicates_id = train_df.duplicated(subset=['ID']).sum()
print(f"Số dòng trùng lặp hoàn toàn: {duplicates_total}")
print(f"Số dòng bị trùng lặp ID: {duplicates_id}")

Số dòng trùng lặp hoàn toàn: 0
Số dòng bị trùng lặp ID: 0


## 7. Khảo sát sơ bộ các giá trị phân loại

Để đảm bảo dữ liệu phân loại sạch, ta kiểm tra các giá trị độc bản (unique values) của từng cột dạng chuỗi.

In [7]:
categorical_cols = train_df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f"Cột {col}: {train_df[col].dropna().unique()}")

Cột Gender: ['Male' 'Female']
Cột Ever_Married: ['No' 'Yes']
Cột Graduated: ['No' 'Yes']
Cột Profession: ['Healthcare' 'Engineer' 'Lawyer' 'Entertainment' 'Artist' 'Executive'
 'Doctor' 'Homemaker' 'Marketing']
Cột Spending_Score: ['Low' 'Average' 'High']
Cột Var_1: ['Cat_4' 'Cat_6' 'Cat_7' 'Cat_3' 'Cat_1' 'Cat_2' 'Cat_5']
Cột Segmentation: ['D' 'A' 'B' 'C']


## 8. Mô tả thống kê sơ bộ (Descriptive Statistics)

Thống kê mô tả đối với cả các cột số học.

In [8]:
train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
ID,8068.0,463479.214551,2595.381232,458982.0,461240.75,463472.5,465744.25,467974.0
Age,8068.0,43.466906,16.711696,18.0,30.00,40.0,53.00,89.0
Work_Experience,7239.0,2.641663,3.406763,0.0,0.00,1.0,4.00,14.0
Family_Size,7733.0,2.850123,1.531413,1.0,2.00,3.0,4.00,9.0
